In [ ]:
# === LIBRARIES AND INSTALLATION REQUIREMENTS ===

import os, time, json
import pandas as pd
from openai import OpenAI
import json, re, time
from string import Template


# === API KEY CONFIGURATION ===

import os; os.environ["OPENROUTER_API_KEY"] = "APIKEY"


# === INITIAL MODEL CONFIGURATION ===

INPUT_CSV  = "YOUR_IMPUT"
OUTPUT_CSV = "YOUR_OUTPUT"
MODELO      = "meta-llama/llama-3.1-8b-instruct"
LIM_GUARDAR = 100
MAX_INTENTOS = 4
SLEEP_ENTRE_CALLS = 0.6
TRUNCATE_CHARS = 3000


# === API KEY DATA CONFIGURATION ===

api_key = os.environ["OPENROUTER_API_KEY"]  # Already validated
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)


# === PROMPT AND INITIAL ERROR HANDLING ===

SYSTEM_MSG = (
    'Respond ONLY with a valid JSON in the form {"keywords":[...]}. '
    'Do not include explanations, markdown, apologies, or code blocks. '

    'STRICT LANGUAGE RULE: '
    '- The output MUST contain ONLY ENGLISH WORDS. '
    '- Translate foreign terms into English; if translation is impossible, use a English descriptive equivalent. '
    '- Before responding, internally verify that EVERY keyword is fully in English. '
)

PROMPT_TMPL = Template("""
You are an expert assistant in bibliographic analysis.
From the following article record (authors, title, year, journal, abstract, original keywords),
extract EXACTLY 5 terms that represent the main concepts of the article.

Rules:
- ALWAYS return: {"keywords": ["term1", "term2", ...]}.
- Output MUST be ONLY in ENGLISH; no other languages under any circumstance.
- Normalize to lowercase, without accents or special characters (ej. "mathematics education").
- Accept short multiword phrases (2–4 words).
- Avoid generic terms such as: "article", "study", "analysis", "research", "work", "keyword".
- Prioritize disciplinary concepts, population, context, method, theory.
- ALWAYS return exactly 5 items in the list; if a concept cannot be expressed in English, use the string "null" as a placeholder, which still counts toward the five terms.

Article text:
$texto

Before responding, SELF-CHECK: "all the keywords are in English without exception?"
""")


# === FUNCTION DEFINITIONS AND ERROR HANDLING ===

# Accepted keys in case the model changes language or uses singular/plural forms
ACCEPT_KEYS = ["keywords", "keyword", "palabras_clave", "palabras", "términos", "terminos"]

# === DICTIONARY EXTRACTION FUNCTION ===

def _extraer_json(texto: str) -> dict | None:
    """Robust JSON parsing with fault tolerance.

    This function attempts to recover a valid JSON object from the outputs
    generated by the LLM. It implements error handling and fallback rules
    for cases in which the output does not strictly follow the expected format.

    Logic
    -----
    1. Attempt to extract and parse a complete JSON object {...}.
    2. If it is not valid, attempt to extract a list [...] and wrap it as
       {"keywords": [...]}.
       - Supports both valid JSON lists and ad hoc lists of quoted strings.
    3. Normalize all strings to lowercase, remove accents, and trim extra spaces.
    4. Return None if no valid structure is found.

    Parameters
    ----------
    texto : str
        Raw text that may contain JSON-like content.

    Returns
    -------
    dict | None
        A dictionary with the structure {"keywords": [...]} if successful,
        or None if a valid JSON structure cannot be recovered.
    """

    texto = str(texto).strip()

    # 1) Attempt to extract an object { ... }
    primero, ultimo = texto.find("{"), texto.rfind("}")
    if primero != -1 and ultimo != -1 and ultimo > primero:
        fragmento = texto[primero:ultimo+1]
        try:
            return json.loads(fragmento)
        except Exception:
            pass  # Use another method

    # 2) Attempt to extract a list [ ... ] --> list of strings
    lb, rb = texto.find("["), texto.rfind("]")
    if lb != -1 and rb != -1 and rb > lb:
        fragmento = texto[lb:rb+1]
        try:
            lista = json.loads(fragmento)
        except Exception:
            # Alternative: extract "..." or '...' strings from the list
            lista = re.findall(r'["\']([^"\']+)["\']', fragmento)
        # Normalize to a list of strings
        lista = [str(x).strip() for x in lista if str(x).strip()]
        return {"Keywords": lista}

    # 3) No useful structure was found
    return None


# === NORMALIZATION FUNCTION ===

def normalizar_lista_palabras(palabras):
    """Receives a list of words and returns a normalized version:
    - All lowercase
    - No accents
    - No extra spaces
    - No duplicates
    """

    if not isinstance(palabras, list):
        return []

    resultado = []
    for palabra in palabras:
        # Convert to string and trim leading and trailing spaces
        s = str(palabra).strip()
        if not s:
            continue

        # Convert to lowercase and remove accents
        s = s.lower()
        s = (s.replace("á", "a")
               .replace("é", "e")
               .replace("í", "i")
               .replace("ó", "o")
               .replace("ú", "u"))

        # Replace multiple spaces with a single space
        s = re.sub(r"\s+", " ", s)

        # Add only if it is not duplicated
        if s not in resultado:
            resultado.append(s)

    return resultado


# === API CALL FUNCTION ===

def call_llm(texto: str) -> dict:
    """
    Calls the LLM and returns a dictionary with the following structure:
      {"ok": bool, "keywords": [str, ...], "raw": str}

    - Uses Template.substitute to avoid KeyError if the text contains braces.
    - Retries transient errors with exponential backoff.
    - Robust parsing: accepts {"keywords":[...]}, other equivalent keys,
      a standalone list, or, as a last resort, quoted or comma-separated terms.
    """

    delay = 1.2
    for intento in range(1, MAX_INTENTOS + 1):
        try:
            # 1) Prompt construction
            user_content = PROMPT_TMPL.substitute(texto=texto)

            # 2) Model call
            resp = client.chat.completions.create(
                model=MODELO,
                messages=[
                    {"role": "system", "content": SYSTEM_MSG},
                    {"role": "user", "content": user_content},
                ],
                temperature=0.0,
                max_tokens=700,
            )

            content = (resp.choices[0].message.content or "").strip()

            # 3) Primary attempt
            parsed = _extraer_json(content)

            # 4) Extract keywords from the dictionary returned by the previous function
            kws = []
            if isinstance(parsed, dict):
                # Accept key variants included in ACCEPT_KEYS
                keys_en_respuesta = set(parsed.keys())
                for k in ACCEPT_KEYS:
                    if k in parsed:
                        kws = parsed[k]
                        break
                    # Edge case: a parser returned keys with quotation marks included
                    quoted_k = f'"{k}"'
                    if quoted_k in keys_en_respuesta:
                        kws = parsed[quoted_k]
                        break
            elif isinstance(parsed, list):
                kws = parsed

            # 5) Normalization
            kws = normalizar_lista_palabras(kws)

            # 6) If empty, attempt to recover quoted or comma-separated terms
            if not kws:
                entrecomillados = re.findall(r'["\']([^"\']+)["\']', content)
                kws = (normalizar_lista_palabras(entrecomillados)
                       or normalizar_lista_palabras([t for t in content.split(",") if t]))

            return {"ok": True, "keywords": kws, "raw": content}

        except Exception as e:
            msg = f"{type(e).__name__}: {e}"
            print(f"intento {intento} falló: {msg}")
            if intento == MAX_INTENTOS:
                return {"ok": False, "keywords": [], "raw": msg}
            time.sleep(delay)
            delay *= 1.8


# === DATA LOADING ===

df = pd.read_csv(INPUT_CSV)
df = df.reset_index().rename(columns={"index": "row_abs"})

# === BATCH CONFIGURATION ===

START_ILOC, STOP_ILOC = 0, 53131
df = df.iloc[START_ILOC:STOP_ILOC].copy()

assert "insumo" in df.columns, "el CSV debe tener la columna insumo"

total = len(df)
primera_abs = int(df["row_abs"].iloc[0]) + 1
ultima_abs   = int(df["row_abs"].iloc[-1]) + 1
print(f"Procesaremos {total} filas: originales {primera_abs} a {ultima_abs} (iloc {START_ILOC}:{STOP_ILOC}).")

# Resume if the output file exists
if os.path.exists(OUTPUT_CSV):
    out = pd.read_csv(OUTPUT_CSV)

    if "row_abs" not in out.columns and "row_index" in out.columns:
        out = out.rename(columns={"row_index": "row_abs"})

    if "row_abs" in out.columns:
        done_idx = set(pd.to_numeric(out["row_abs"], errors="coerce").dropna().astype(int).tolist())
    else:
        out = out.reset_index().rename(columns={"index": "row_abs"})
        done_idx = set(pd.to_numeric(out["row_abs"], errors="coerce").dropna().astype(int).tolist())

    print(f"Reanudando desde {len(done_idx)} ya procesadas.")
else:
    out = pd.DataFrame(columns=["row_abs", "keywords_llm", "raw_response"])
    done_idx = set()

buffer_rows, processed_since_save = [], 0

for _, row in df[["row_abs", "insumo"]].iterrows():
    rid = int(row["row_abs"])
    if rid in done_idx:
        continue

    texto = str(row["insumo"])[:TRUNCATE_CHARS]
    res = call_llm(texto)

    if not res["ok"]:
        print(f"fila abs {rid} error: {res['raw']}")

    # Store keywords as JSON
    keywords_json = json.dumps(res.get("keywords", []), ensure_ascii=False)
    raw = res.get("raw", "")

    buffer_rows.append({"row_abs": rid, "keywords_llm": keywords_json, "raw_response": raw})
    processed_since_save += 1

    if (processed_since_save % LIM_GUARDAR == 0):
        chunk = pd.DataFrame(buffer_rows)
        merged = pd.concat([out, chunk], ignore_index=True)
        merged.drop_duplicates(subset=["row_abs"], keep="last", inplace=True)
        merged.sort_values("row_abs", inplace=True)

        tmp_path = OUTPUT_CSV + ".tmp"
        merged.to_csv(tmp_path, index=False)
        os.replace(tmp_path, OUTPUT_CSV)

        print(f"Guardado parcial: {len(merged)} / {total}")
        out = merged
        buffer_rows, processed_since_save = [], 0

    time.sleep(SLEEP_ENTRE_CALLS)

# Final flush
if buffer_rows:
    chunk = pd.DataFrame(buffer_rows)
    merged = pd.concat([out, chunk], ignore_index=True)
    merged.drop_duplicates(subset=["row_abs"], keep="last", inplace=True)
    merged.sort_values("row_abs", inplace=True)

    tmp_path = OUTPUT_CSV + ".tmp"
    merged.to_csv(tmp_path, index=False)
    os.replace(tmp_path, OUTPUT_CSV)

    print(f"Guardado final: {len(merged)} / {total}")

print("Terminado. Archivo:", OUTPUT_CSV)